# Encoding — Titanic Dataset

This notebook continues from `titanic_outliers_handled.csv`.

We will identify categorical features, understand nominal vs ordinal data, apply one-hot encoding, understand `drop_first=True`, and save `titanic_encoded.csv` for the next Scaling step.

## 1. Import Libraries

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 2. Load the Dataset

Place `titanic_outliers_handled.csv` in the same directory as this notebook.

In [ ]:
df = pd.read_csv('../Dataset/02_titanic_outliers_handled.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


## 3. Inspect the Data

In [3]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nUnique values:')
display(df.nunique().sort_values())

Shape: (891, 11)

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
dtype: object

Unique values:


Survived         2
Sex              2
Pclass           3
Embarked         3
Parch            7
SibSp            7
Age             88
Fare           248
Ticket         681
Name           891
PassengerId    891
dtype: int64

## 4. Identify Categorical Features

For this dataset, `Sex` and `Embarked` are nominal categorical features.

`Pclass` is an ordered category but is already represented numerically as 1, 2, and 3.

`Survived` is the target and should not be one-hot encoded as an input feature.

`PassengerId` is an identifier, not a meaningful feature.

In [4]:
categorical_features = ['Sex', 'Embarked']
df[categorical_features].head()

,Sex,Embarked
0,male,S
1,female,C
2,female,S
3,female,S
4,male,S


## 5. Nominal vs Ordinal

**Nominal:** categories have no natural order, e.g. `Sex` and `Embarked`.

**Ordinal:** categories have a meaningful order, e.g. passenger class: 1st < 2nd < 3rd.

One-hot encoding is appropriate for the nominal features here.

## 6. One-Hot Encoding

`pd.get_dummies()` creates a binary column for each category.

In [5]:
df_one_hot = pd.get_dummies(df, columns=categorical_features, dtype=int)
df_one_hot.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,0,1,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,1,0,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1,0,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,1,0,0,0,1
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,0,1,0,0,1


The original `Sex` and `Embarked` columns are replaced with columns such as `Sex_female`, `Sex_male`, `Embarked_C`, `Embarked_Q`, and `Embarked_S`.

## 7. Understanding `drop_first=True`

If a feature has `k` categories, one-hot encoding creates `k` dummy columns. With `drop_first=True`, one category is removed because it is represented when all remaining dummy columns are 0.

This can reduce redundant dummy variables, particularly for linear models.

In [6]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

df_encoded.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,1,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,0,0,0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,0,0,1
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,1,0,1


## 8. Compare Before and After

In [7]:
print('Original columns:')
print(df.columns.tolist())

print('\nEncoded columns:')
print(df_encoded.columns.tolist())

print('\nOriginal shape:', df.shape)
print('Encoded shape:', df_encoded.shape)

Original columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']

Encoded columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']

Original shape: (891, 11)
Encoded shape: (891, 12)


## 9. Verify the Encoding

In [8]:
encoded_columns = [
    col for col in df_encoded.columns
    if col.startswith('Sex_') or col.startswith('Embarked_')
]

display(df_encoded[encoded_columns].head())

print('Categorical columns still present:',
      [col for col in categorical_features if col in df_encoded.columns])

,Sex_male,Embarked_Q,Embarked_S
0,1,0,1
1,0,0,0
2,0,0,1
3,0,0,1
4,1,0,1


Categorical columns still present: []


## 10. Do Not Encode Everything

| Column | Treatment |
|---|---|
| `PassengerId` | Identifier; remove before modeling |
| `Survived` | Target; keep as target |
| `Pclass` | Already numerical and ordinal |
| `Age` | Numerical |
| `SibSp` | Numerical/count |
| `Parch` | Numerical/count |
| `Fare` | Numerical |
| `Sex` | One-hot encode |
| `Embarked` | One-hot encode |

## 11. Remove the Identifier

`PassengerId` is an identifier and normally should not be used as a model input.

In [9]:
df_encoded = df_encoded.drop(columns=['PassengerId'])
df_encoded.head()

,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,1,0,1
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,0,0,0
2,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,0,0,1
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,0,0,1
4,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,1,0,1


## 12. Final Verification

In [10]:
print('Shape:', df_encoded.shape)
print('\nMissing values:')
print(df_encoded.isnull().sum())
print('\nData types:')
print(df_encoded.dtypes)

Shape: (891, 11)

Missing values:
Survived      0
Pclass        0
Name          0
Age           0
SibSp         0
Parch         0
Ticket        0
Fare          0
Sex_male      0
Embarked_Q    0
Embarked_S    0
dtype: int64

Data types:
Survived        int64
Pclass          int64
Name              str
Age           float64
SibSp           int64
Parch           int64
Ticket            str
Fare          float64
Sex_male        int64
Embarked_Q      int64
Embarked_S      int64
dtype: object


## 13. Save the Encoded Dataset

This becomes the input for the next preprocessing stage: **Scaling**.

In [ ]:
output_path = '../Dataset/03_titanic_encoded.csv'
df_encoded.to_csv(output_path, index=False)
print(f'Saved: {output_path}')

Saved: ../Dataset/titanic_encoded.csv


## Key Takeaways

- Encoding converts categorical features into numerical representations.
- Nominal features are commonly handled with one-hot encoding.
- `pd.get_dummies()` is a simple way to apply one-hot encoding.
- `drop_first=True` removes one redundant dummy column.
- Do not blindly encode identifiers or the target.
- `Pclass` is already an ordered numerical feature.

### Next

```text
Missing Values
      ↓
Outliers
      ↓
Encoding
      ↓
Scaling
      ↓
Feature Engineering
      ↓
ML-Ready Dataset
```